# The position battery — wins, defence, and openings

The deployed engine must find forced wins in one to three turns, defend against threats
on the same horizons, and play consistent, survivable lines from the start. This notebook
is the executable record of that claim: it replays the nineteen curated positions of
`benchmarks/battery.json` through the battery runner in `tuning/harness.py` — the engine
playing itself in the deployment configuration (opening book, depth 3, quiescence,
forced-win layer) — and **asserts every verdict, ply count and node budget** against the
documented table of
[docs/EVALUATION.md, section 11](../docs/EVALUATION.md).

The entries are independent self-plays, so the battery runs on every core at once
(`os.cpu_count()` worker processes); only wall times are machine-dependent — verdicts,
plies and nodes are deterministic, and each entry is played twice with both transcripts
required identical. Node counts asserted here were measured sequentially; matching them
from the parallel run is itself the parallel-equals-sequential check.

In [1]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))                # connect6
sys.path.insert(0, str(PROJECT_ROOT / "tuning"))    # harness
import harness

battery = json.loads((PROJECT_ROOT / "benchmarks" / "battery.json").read_text())
print(f"{len(battery)} positions, {os.cpu_count()} cores available")
print(f"weights: {os.environ.get('WEIGHTS_FILE', 'tuning/winner.json (default)')}, "
      f"book: {os.environ.get('BOOK_FILE', 'tuning/opening_book.json (default)')}")

19 positions, 12 cores available
weights: tuning/winner.json (default), book: tuning/opening_book.json (default)


In [2]:
import time

jobs = os.cpu_count()
start = time.perf_counter()
results = harness.run_battery(battery, jobs)
wall = time.perf_counter() - start
print(f"parallel run: {len(results)} entries on {jobs} jobs in {wall:.1f}s")

parallel run: 19 entries on 12 jobs in 309.2s


In [3]:
from connect6.defines import Color

rows = []
for entry, result in zip(battery, results):
    winner = "-" if result["winner"] is None else ("black" if result["winner"] == Color.BLACK else "white")
    rows.append((entry["id"], result["class"], winner, result["plies"], result["nodes"], result["verdict"]))
print(f"{'id':<36} {'class':<7} {'winner':<6} {'plies':<5} {'nodes':<8} verdict")
for r in rows:
    print(f"{r[0]:<36} {r[1]:<7} {r[2]:<6} {r[3]:<5} {r[4]:<8} {'PASS' if r[5] else 'FAIL'}")

id                                   class   winner plies nodes    verdict
win-1-horizontal-live-five           win     black  1     1        PASS
win-1-diagonal-live-five             win     black  1     1        PASS
defend-1-horizontal-live-five        defend  -      2     11061    PASS
defend-1-diagonal-live-five          defend  -      2     31170    PASS
win-1-seedJJ                         win     white  1     1        PASS
win-2-seedJJ                         win     white  3     1732     PASS
win-3-seedJJ                         win     white  5     1917     PASS
win-2-seedKJIJ                       win     white  3     16124    PASS
win-3-seedHIKK                       win     white  5     47811    PASS
win-1-seedMNMN                       win     white  1     1        PASS
defend-3-opening-fork                defend  -      6     150157   PASS
defend-2-early-fork                  defend  -      4     49355    PASS
defend-1-late-pressure               defend  -      2     315

In [4]:
# The documented table of EVALUATION.md section 11: id -> (plies, nodes).
EXPECTED = {
    "win-1-horizontal-live-five": (1, 1),
    "win-1-diagonal-live-five": (1, 1),
    "win-1-seedJJ": (1, 1),
    "win-1-seedMNMN": (1, 1),
    "win-2-seedJJ": (3, 1732),
    "win-2-seedKJIJ": (3, 16124),
    "win-3-seedJJ": (5, 1917),
    "win-3-seedHIKK": (5, 47811),
    "defend-1-horizontal-live-five": (2, 11061),
    "defend-1-diagonal-live-five": (2, 31170),
    "defend-1-late-pressure": (2, 3155),
    "defend-1-live-four": (2, 95758),
    "defend-2-early-fork": (4, 49355),
    "defend-2-double-threat": (4, 254424),
    "defend-3-opening-fork": (6, 150157),
    "defend-3-fork-build": (6, 222772),
    "start-empty-black-survives-opening": (20, 92336),
    "start-empty-white-converts-opening": (20, 92336),
    "start-kjij-white-converts": (12, 215634),
}

assert all(result["verdict"] for result in results), "a battery entry failed its verdict"
assert all(result["consistent"] for result in results), "an entry's two transcripts differ"
for entry, result in zip(battery, results):
    plies, nodes = EXPECTED[entry["id"]]
    assert result["plies"] == plies, (entry["id"], "plies", result["plies"], plies)
    assert result["nodes"] == nodes, (entry["id"], "nodes", result["nodes"], nodes)
print(f"all {len(EXPECTED)} verdicts, transcript identities, ply counts and node budgets asserted")

all 19 verdicts, transcript identities, ply counts and node budgets asserted


**What a clean run means.** Every win entry converted inside its horizon, every defence
held it, the opening lines reached their documented lengths, and the parallel run
reproduced the sequential node budgets exactly — the battery, the docs and the engine
agree. A FAIL or a node diff is a behaviour change: that is the battery's job.